# DNMR Google-Drive-Ready Notebook

This notebook is updated for datasets stored in **Google Drive / Colab**.

Supported dataset files:
- `ml-1m.zip` (MovieLens 1M ratings)
- `reviews_Beauty_5.json.gz` (Amazon Beauty reviews, JSON-lines)
- `Sports_and_Outdoors.jsonl.gz` (Amazon Sports reviews, JSON-lines)

Workflow:
1. mount Google Drive
2. point each dataset to its file path in Drive
3. extract ml-1m zip; use .json.gz files directly
4. run SASRec, DNMR-MemoryOnly, and DNMR-Full-v2
5. summarize results across all datasets

In [2]:
import math
import json
import random
import zipfile
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

## 1. Mount Google Drive in Colab

In [9]:
# Run this cell in Google Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [8]:
@dataclass
class Config:
    max_len: int = 50
    embed_dim: int = 64
    proj_dim: int = 64
    num_layers: int = 2
    num_heads: int = 2
    dropout: float = 0.2
    padding_idx: int = 0

    batch_size: int = 256
    lr: float = 1e-3
    weight_decay: float = 1e-6
    epochs: int = 30

    num_eval_negatives: int = 100

    memory_size: int = 1024
    temperature: float = 0.2
    push_threshold: float = 0.5
    reliability_floor: float = 0.3
    ambiguity_topk: int = 64
    user_alpha: float = 0.1

    use_memory: bool = True
    use_user_conditioned: bool = True
    hard_filter: bool = False

    warmup_epochs: int = 5
    early_stopping_patience: int = 5

cfg = Config()
cfg

Config(max_len=50, embed_dim=64, proj_dim=64, num_layers=2, num_heads=2, dropout=0.2, padding_idx=0, batch_size=256, lr=0.001, weight_decay=1e-06, epochs=30, num_eval_negatives=100, memory_size=1024, temperature=0.2, push_threshold=0.5, reliability_floor=0.3, ambiguity_topk=64, user_alpha=0.1, use_memory=True, use_user_conditioned=True, hard_filter=False, warmup_epochs=5, early_stopping_patience=5)

## 2. Google Drive dataset paths

In [10]:
# Set these to your dataset files in Google Drive
ML1M_ZIP_PATH = "/content/drive/MyDrive/DNMR/ml-1m.zip"   # @param {type:"string"}
AMAZON_BEAUTY_PATH = "/content/drive/MyDrive/DNMR/Beauty_and_Personal_Care.jsonl.gz"   # @param {type:"string"}
AMAZON_SPORTS_PATH = "/content/drive/MyDrive/DNMR/Sports_and_Outdoors.jsonl.gz"   # @param {type:"string"}

# Local extraction root in Colab (only used for ml-1m zip)
EXTRACT_ROOT = "./dataset"

POSITIVE_RATING_THRESHOLD = 4.0
MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 5

## 3. Extract ml-1m and resolve dataset paths

In [12]:
def extract_zip_from_drive(zipped_file_path: str, extract_dir: str) -> str:
    """Extract a zip file into extract_dir."""
    if not zipped_file_path:
        raise ValueError("Please specify a valid zipped file path.")

    print(f"Using zipped file from Google Drive path: {zipped_file_path}")
    os.makedirs(extract_dir, exist_ok=True)

    with zipfile.ZipFile(zipped_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    print(f"Files unzipped to: {extract_dir}")
    return extract_dir


def find_ratings_dat(extract_dir: str) -> str:
    """Walk extract_dir to find ratings.dat for MovieLens."""
    for root, dirs, files in os.walk(extract_dir):
        for f in files:
            if f.lower() == 'ratings.dat':
                return os.path.join(root, f)
    raise FileNotFoundError(f"Could not find ratings.dat inside {extract_dir}")


def prepare_dataset_paths(
    ml1m_zip_path: str,
    beauty_path: str,
    sports_path: str,
    extract_root: str = "./dataset",
):
    """Resolve paths for all three datasets.

    - ml-1m: extract the zip, locate ratings.dat inside.
    - beauty / sports: use the .json.gz paths directly from Drive.
    """
    extract_root = Path(extract_root)
    ml_dir = extract_root / "ml1m"
    extract_zip_from_drive(ml1m_zip_path, str(ml_dir))
    ml_file = find_ratings_dat(str(ml_dir))

    # Validate that the beauty/sports files exist on Drive
    for label, p in [("Amazon Beauty", beauty_path), ("Amazon Sports", sports_path)]:
        if not os.path.isfile(p):
            raise FileNotFoundError(
                f"{label} file not found at: {p}\n"
                f"Expected a .json.gz or .json file with review data."
            )

    print("Detected MovieLens data file path:", ml_file)
    print("Detected Amazon Beauty data file path:", beauty_path)
    print("Detected Amazon Sports data file path:", sports_path)

    return {
        "ml1m": ml_file,
        "amazon_beauty": beauty_path,
        "amazon_sports": sports_path,
    }


DATA_PATHS = prepare_dataset_paths(
    ML1M_ZIP_PATH,
    AMAZON_BEAUTY_PATH,
    AMAZON_SPORTS_PATH,
    EXTRACT_ROOT,
)

Using zipped file from Google Drive path: /content/drive/MyDrive/DNMR/ml-1m.zip
Files unzipped to: dataset/ml1m
Detected MovieLens data file path: dataset/ml1m/ml-1m/ratings.dat
Detected Amazon Beauty data file path: /content/drive/MyDrive/DNMR/Beauty_and_Personal_Care.jsonl.gz
Detected Amazon Sports data file path: /content/drive/MyDrive/DNMR/Sports_and_Outdoors.jsonl.gz


## 4. Dataset loaders

In [14]:
def _build_sequences_from_events(raw_events, min_user_interactions=5, min_item_interactions=5):
    changed = True
    while changed:
        changed = False
        uc, ic = {}, {}
        for u, i, _ in raw_events:
            uc[u] = uc.get(u, 0) + 1
            ic[i] = ic.get(i, 0) + 1

        filtered = [
            (u, i, t)
            for (u, i, t) in raw_events
            if uc[u] >= min_user_interactions and ic[i] >= min_item_interactions
        ]
        if len(filtered) != len(raw_events):
            changed = True
            raw_events = filtered

    histories = {}
    for u, i, t in raw_events:
        histories.setdefault(u, []).append((t, i))

    cleaned = {}
    for u, ev in histories.items():
        ev.sort(key=lambda x: x[0])
        seq, last_i = [], None
        for _, i in ev:
            if i != last_i:
                seq.append(i)
            last_i = i
        if len(seq) >= min_user_interactions:
            cleaned[u] = seq

    orig_users = sorted(cleaned.keys())
    orig_items = sorted({i for seq in cleaned.values() for i in seq})

    user_id_map = {u: idx + 1 for idx, u in enumerate(orig_users)}
    item_id_map = {i: idx + 1 for idx, i in enumerate(orig_items)}

    user_sequences = {
        user_id_map[u]: [item_id_map[i] for i in seq]
        for u, seq in cleaned.items()
    }
    return user_sequences, len(item_id_map), user_id_map, item_id_map


def load_movielens_1m(ratings_path, min_user_interactions=5, min_item_interactions=5, positive_rating_threshold=4.0):
    raw_events = []
    with open(ratings_path, "r", encoding="latin-1") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            u, i, r, ts = line.split("::")
            if float(r) >= positive_rating_threshold:
                raw_events.append((str(u), str(i), int(ts)))
    if not raw_events:
        raise ValueError("No positive interactions found in MovieLens.")
    return _build_sequences_from_events(raw_events, min_user_interactions, min_item_interactions)


def load_amazon_reviews(reviews_path, min_user_interactions=5, min_item_interactions=5, positive_rating_threshold=4.0, verified_only=True):
    """Load Amazon review JSON-lines (.json or .json.gz / .jsonl.gz).

    Supports two schemas:
      Legacy (e.g. reviews_Beauty_5.json.gz):
        reviewerID, asin, overall, unixReviewTime
      2023 format (e.g. Sports_and_Outdoors.jsonl.gz):
        user_id, asin/parent_asin, rating, timestamp
    """
    raw_events = []
    open_fn = open
    mode = "r"
    kwargs = {"encoding": "utf-8"}

    if str(reviews_path).lower().endswith(".gz"):
        import gzip
        open_fn = gzip.open
        mode = "rt"

    is_2023 = None  # auto-detect on first valid line

    with open_fn(reviews_path, mode, **kwargs) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            # Auto-detect schema on first record
            if is_2023 is None:
                is_2023 = "user_id" in obj
                fmt = "2023" if is_2023 else "legacy"
                print(f"Detected {fmt} Amazon review format in {os.path.basename(reviews_path)}")

            if is_2023:
                # 2023 format: user_id, asin, rating, timestamp
                if verified_only and not obj.get("verified_purchase", False):
                    continue
                user = obj.get("user_id")
                item = obj.get("asin") or obj.get("parent_asin")
                rating = float(obj.get("rating", 0))
                # 2023 timestamps are in milliseconds
                ts_raw = obj.get("timestamp", 0)
                ts = int(ts_raw) // 1000 if int(ts_raw) > 1e12 else int(ts_raw)
            else:
                # Legacy format: reviewerID, asin, overall, unixReviewTime
                user = obj.get("reviewerID")
                item = obj.get("asin")
                rating = float(obj.get("overall", 0))
                ts = int(obj.get("unixReviewTime", 0))

            if not user or not item:
                continue
            if rating >= positive_rating_threshold:
                raw_events.append((str(user), str(item), ts))

    if not raw_events:
        raise ValueError(f"No positive interactions found in {reviews_path}")
    print(f"  Loaded {len(raw_events)} positive interactions from {os.path.basename(reviews_path)}")
    return _build_sequences_from_events(raw_events, min_user_interactions, min_item_interactions)


def load_dataset_by_name(dataset_name, data_paths, min_user_interactions=5, min_item_interactions=5, positive_rating_threshold=4.0):
    if dataset_name == "ml1m":
        return load_movielens_1m(
            data_paths["ml1m"], min_user_interactions,
            min_item_interactions, positive_rating_threshold,
        )
    elif dataset_name in ("amazon_beauty", "amazon_sports"):
        return load_amazon_reviews(
            data_paths[dataset_name], min_user_interactions,
            min_item_interactions, positive_rating_threshold,
        )
    raise ValueError(f"Unsupported dataset_name: {dataset_name}")

## 5. Split, datasets, metrics, model, memory, training, and experiment runners

In [15]:
def leave_one_out_split(user_sequences: Dict[int, List[int]]):
    train, val, test = {}, {}, {}
    for u, seq in user_sequences.items():
        if len(seq) < 3:
            continue
        train[u], val[u], test[u] = seq[:-2], seq[-2], seq[-1]
    return train, val, test

def pad_sequence(seq: List[int], max_len: int, pad_value: int = 0):
    seq = seq[-max_len:]
    return [pad_value] * (max_len - len(seq)) + seq

def sample_negative(num_items: int, forbidden: set, max_tries: int = 10000):
    """Sample a random item not in forbidden. Raises if num_items is exhausted."""
    if len(forbidden) >= num_items:
        raise RuntimeError(
            f"Cannot sample a negative: all {num_items} items are forbidden."
        )
    for _ in range(max_tries):
        item = random.randint(1, num_items)
        if item not in forbidden:
            return item
    # Fallback: deterministic scan (very unlikely to reach here)
    for item in range(1, num_items + 1):
        if item not in forbidden:
            return item

class SequentialTrainDataset(Dataset):
    """Negatives are re-sampled every __getitem__ call for diversity."""
    def __init__(self, train_sequences, num_items, max_len):
        self.num_items = num_items
        self.samples = []  # (user_id, padded_hist, pos_item, forbidden_set)
        for u, seq in train_sequences.items():
            if len(seq) < 2:
                continue
            for t in range(1, len(seq)):
                hist, pos = seq[:t], seq[t]
                padded = pad_sequence(hist, max_len)
                forbidden = set(hist) | {pos}
                self.samples.append((u, padded, pos, forbidden))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        u, hist, pos, forbidden = self.samples[idx]
        neg = sample_negative(self.num_items, forbidden)
        return torch.tensor(u), torch.tensor(hist), torch.tensor(pos), torch.tensor(neg)

class SequentialEvalDataset(Dataset):
    def __init__(self, train_sequences, target_items, num_items, max_len, num_negatives=100):
        self.samples = []
        for u, train_seq in train_sequences.items():
            if u not in target_items:
                continue
            pos = target_items[u]
            padded = pad_sequence(train_seq, max_len)
            forbidden = set(train_seq) | {pos}
            negatives = []
            while len(negatives) < num_negatives:
                item = random.randint(1, num_items)
                if item not in forbidden:
                    negatives.append(item)
                    forbidden.add(item)
            self.samples.append((u, padded, pos, negatives))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        u, hist, pos, negatives = self.samples[idx]
        return torch.tensor(u), torch.tensor(hist), torch.tensor(pos), torch.tensor(negatives)

def hit_rate_at_k(rank, k): return 1.0 if rank < k else 0.0
def ndcg_at_k(rank, k): return 1.0 / math.log2(rank + 2.0) if rank < k else 0.0

class SASRecBlock(nn.Module):
    def __init__(self, dim, num_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.ln1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, dim * 4), nn.ReLU(), nn.Dropout(dropout), nn.Linear(dim * 4, dim))
        self.ln2 = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, attn_mask=None):
        attn_out, _ = self.attn(x, x, x, attn_mask=attn_mask, need_weights=False)
        x = self.ln1(x + self.dropout(attn_out))
        ffn_out = self.ffn(x)
        x = self.ln2(x + self.dropout(ffn_out))
        return x

class DNMR(nn.Module):
    def __init__(self, num_items, cfg):
        super().__init__()
        self.cfg = cfg
        self.item_embedding = nn.Embedding(num_items + 1, cfg.embed_dim, padding_idx=cfg.padding_idx)
        self.pos_embedding = nn.Embedding(cfg.max_len, cfg.embed_dim)
        self.dropout = nn.Dropout(cfg.dropout)
        self.layers = nn.ModuleList([SASRecBlock(cfg.embed_dim, cfg.num_heads, cfg.dropout) for _ in range(cfg.num_layers)])
        self.projection = nn.Sequential(nn.Linear(cfg.embed_dim, cfg.embed_dim), nn.ReLU(), nn.Linear(cfg.embed_dim, cfg.proj_dim))
    def encode_user(self, seq):
        bsz, seq_len = seq.shape
        pos_ids = torch.arange(seq_len, device=seq.device).unsqueeze(0).expand(bsz, -1)
        x = self.item_embedding(seq) + self.pos_embedding(pos_ids)
        x = self.dropout(x)
        causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=seq.device, dtype=torch.bool), diagonal=1)
        for layer in self.layers:
            x = layer(x, attn_mask=causal_mask)
        lengths = (seq != self.cfg.padding_idx).sum(dim=1).clamp(min=1) - 1
        return x[torch.arange(bsz, device=seq.device), lengths]
    def project_item(self, item_emb): return F.normalize(self.projection(item_emb), dim=-1)
    def score_items(self, user_repr, item_ids):
        item_emb = self.item_embedding(item_ids)
        if item_emb.dim() == 2:
            score = (user_repr * item_emb).sum(dim=-1)
        else:
            score = (user_repr.unsqueeze(1) * item_emb).sum(dim=-1)
        return score, item_emb
    def forward(self, seq, pos_items, neg_items):
        user_repr = self.encode_user(seq)
        pos_score, pos_emb = self.score_items(user_repr, pos_items)
        neg_score, neg_emb = self.score_items(user_repr, neg_items)
        neg_proj = self.project_item(neg_emb)
        return {"user_repr": user_repr, "pos_score": pos_score, "neg_score": neg_score, "pos_emb": pos_emb, "neg_emb": neg_emb, "neg_proj": neg_proj}

class DynamicNegativeMemory:
    def __init__(self, capacity, proj_dim, device):
        self.capacity = capacity
        self.proj_dim = proj_dim
        self.device = device
        self.memory = torch.empty((0, proj_dim), device=device)
        self.confidence = torch.empty((0,), device=device)
    def is_empty(self):
        return self.memory.size(0) == 0
    @torch.no_grad()
    def ambiguity(self, neg_proj, temperature=0.2, topk=64):
        if self.is_empty():
            return torch.zeros(neg_proj.size(0), device=neg_proj.device)
        sim = neg_proj @ self.memory.T
        k = min(topk, sim.size(1))
        topk_sim, _ = torch.topk(sim, k=k, dim=1)
        weights = F.softmax(topk_sim / temperature, dim=1)
        ambiguity = (weights * topk_sim).sum(dim=1)
        return ambiguity.clamp(min=0.0, max=1.0)
    @torch.no_grad()
    def update(self, neg_proj, reliability, threshold=0.5):
        keep = reliability >= threshold
        if keep.sum() == 0:
            return
        self.memory = torch.cat([self.memory, neg_proj[keep].detach()], dim=0)
        self.confidence = torch.cat([self.confidence, reliability[keep].detach()], dim=0)
        if self.memory.size(0) > self.capacity:
            overflow = self.memory.size(0) - self.capacity
            self.memory = self.memory[overflow:]
            self.confidence = self.confidence[overflow:]

def compute_reliability(cfg, ambiguity, user_repr, neg_emb):
    base = 1.0 - ambiguity
    if not cfg.use_memory:
        return torch.ones(user_repr.size(0), device=user_repr.device)
    if not cfg.use_user_conditioned:
        return base.clamp(min=cfg.reliability_floor, max=1.0)
    user_affinity = torch.sigmoid((user_repr * neg_emb).sum(dim=-1))
    reliability = base * (1.0 - cfg.user_alpha * user_affinity)
    return reliability.clamp(min=cfg.reliability_floor, max=1.0)

def compute_reliability_raw(cfg, ambiguity, user_repr, neg_emb):
    """Same as compute_reliability but without the floor clamp.

    Used for the memory-update gate so that the reliability_floor
    (which is intentionally low to keep gradients flowing) does not
    prevent negatives from exceeding push_threshold.
    """
    base = 1.0 - ambiguity
    if not cfg.use_memory:
        return torch.ones(user_repr.size(0), device=user_repr.device)
    if not cfg.use_user_conditioned:
        return base.clamp(min=0.0, max=1.0)
    user_affinity = torch.sigmoid((user_repr * neg_emb).sum(dim=-1))
    reliability = base * (1.0 - cfg.user_alpha * user_affinity)
    return reliability.clamp(min=0.0, max=1.0)

def dnmr_loss(cfg, pos_score, neg_score, reliability):
    base = -F.logsigmoid(pos_score - neg_score)
    if cfg.hard_filter:
        keep = (reliability >= cfg.push_threshold).float()
        denom = keep.sum().clamp(min=1.0)
        return (keep * base).sum() / denom
    return (reliability * base).mean()

@torch.no_grad()
def evaluate(model, loader, device, ks=(5, 10, 20)):
    model.eval()
    metrics = {f"HR@{k}": [] for k in ks}
    metrics.update({f"NDCG@{k}": [] for k in ks})
    for _, seq, pos, negatives in loader:
        seq, pos, negatives = seq.to(device), pos.to(device), negatives.to(device)
        user_repr = model.encode_user(seq)
        pos_scores, _ = model.score_items(user_repr, pos)
        neg_scores, _ = model.score_items(user_repr, negatives)
        all_scores = torch.cat([pos_scores.unsqueeze(1), neg_scores], dim=1)
        ranks = torch.argsort(all_scores, dim=1, descending=True)
        rank_positions = (ranks == 0).nonzero(as_tuple=False)[:, 1]
        for r in rank_positions.tolist():
            for k in ks:
                metrics[f"HR@{k}"].append(hit_rate_at_k(r, k))
                metrics[f"NDCG@{k}"].append(ndcg_at_k(r, k))
    return {m: float(np.mean(v)) for m, v in metrics.items()}

def train_one_epoch(model, loader, optimizer, memory, cfg, device, update_memory=True):
    model.train()
    total_loss = total_reliability = total_ambiguity = total_accept_ratio = 0.0
    total_count = 0
    for _, seq, pos, neg in loader:
        seq, pos, neg = seq.to(device), pos.to(device), neg.to(device)
        out = model(seq, pos, neg)
        if cfg.use_memory and memory is not None:
            ambiguity = memory.ambiguity(out["neg_proj"], temperature=cfg.temperature, topk=cfg.ambiguity_topk)
        else:
            ambiguity = torch.zeros_like(out["neg_score"])
        reliability = compute_reliability(cfg, ambiguity, out["user_repr"], out["neg_emb"])
        loss = dnmr_loss(cfg, out["pos_score"], out["neg_score"], reliability)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if cfg.use_memory and memory is not None and update_memory:
            # Use unclamped reliability for the memory gate so the
            # reliability_floor doesn't prevent new entries from being
            # accepted (floor < push_threshold would stall the queue).
            raw_reliability = compute_reliability_raw(cfg, ambiguity, out["user_repr"], out["neg_emb"])
            memory.update(out["neg_proj"], raw_reliability, threshold=cfg.push_threshold)
        accepted_ratio = (reliability >= cfg.push_threshold).float().mean().item()
        bsz = seq.size(0)
        total_loss += loss.item()
        total_reliability += reliability.mean().item() * bsz
        total_ambiguity += ambiguity.mean().item() * bsz
        total_accept_ratio += accepted_ratio * bsz
        total_count += bsz
    return {
        "loss": total_loss / max(1, len(loader)),
        "avg_reliability": total_reliability / max(1, total_count),
        "avg_ambiguity": total_ambiguity / max(1, total_count),
        "accept_ratio": total_accept_ratio / max(1, total_count),
        "queue_size": 0 if memory is None else int(memory.memory.size(0)),
    }

def fit(model, train_loader, val_loader, cfg, device):
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    memory = DynamicNegativeMemory(cfg.memory_size, cfg.proj_dim, device)
    history, best_val, best_state, patience_counter = [], -1.0, None, 0
    for epoch in range(1, cfg.epochs + 1):
        original_use_memory = cfg.use_memory
        in_warmup = epoch <= cfg.warmup_epochs
        if in_warmup:
            cfg.use_memory = False
        train_stats = train_one_epoch(model, train_loader, optimizer, memory, cfg, device, update_memory=not in_warmup)
        cfg.use_memory = original_use_memory
        val_metrics = evaluate(model, val_loader, device)
        history.append({"epoch": epoch, **train_stats, **val_metrics})
        print(
            f"Epoch {epoch:02d} | loss={train_stats['loss']:.4f} | rel={train_stats['avg_reliability']:.4f} | "
            f"amb={train_stats['avg_ambiguity']:.4f} | accept={train_stats['accept_ratio']:.4f} | "
            f"queue={train_stats['queue_size']} | Val NDCG@10={val_metrics['NDCG@10']:.4f} | Val HR@10={val_metrics['HR@10']:.4f}"
        )
        if val_metrics["NDCG@10"] > best_val:
            best_val = val_metrics["NDCG@10"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= cfg.early_stopping_patience:
            print(f"Early stopping triggered at epoch {epoch}.")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return history

def build_dataloaders_for_dataset(dataset_name, cfg, data_paths):
    user_sequences, num_items, user_id_map, item_id_map = load_dataset_by_name(
        dataset_name, data_paths, MIN_USER_INTERACTIONS, MIN_ITEM_INTERACTIONS, POSITIVE_RATING_THRESHOLD
    )
    train_sequences, val_targets, test_targets = leave_one_out_split(user_sequences)
    train_dataset = SequentialTrainDataset(train_sequences, num_items, cfg.max_len)
    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
    val_dataset = SequentialEvalDataset(train_sequences, val_targets, num_items, cfg.max_len, cfg.num_eval_negatives)
    test_dataset = SequentialEvalDataset(
        {u: train_sequences[u] + [val_targets[u]] for u in train_sequences if u in val_targets},
        test_targets, num_items, cfg.max_len, cfg.num_eval_negatives
    )
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)
    print(f"[{dataset_name}] Train examples={len(train_dataset)}, val users={len(val_dataset)}, test users={len(test_dataset)}")
    return {"dataset_name": dataset_name, "num_items": num_items, "train_loader": train_loader, "val_loader": val_loader, "test_loader": test_loader}

def run_setting(name, base_cfg, num_items, train_loader, val_loader, test_loader, device, use_memory, use_user_conditioned, hard_filter):
    local_cfg = Config(**vars(base_cfg))
    local_cfg.use_memory = use_memory
    local_cfg.use_user_conditioned = use_user_conditioned
    local_cfg.hard_filter = hard_filter
    model = DNMR(num_items=num_items, cfg=local_cfg).to(device)
    history = fit(model, train_loader, val_loader, local_cfg, device)
    test_metrics = evaluate(model, test_loader, device)
    print(f"\n{name} TEST METRICS")
    print(test_metrics)
    return {"name": name, "config": local_cfg, "history": history, "test_metrics": test_metrics}

def run_all_methods_on_dataset(dataset_name, cfg, data_paths, device):
    bundle = build_dataloaders_for_dataset(dataset_name, cfg, data_paths)
    num_items = bundle["num_items"]
    tl, vl, tsl = bundle["train_loader"], bundle["val_loader"], bundle["test_loader"]
    return [
        run_setting("SASRec", cfg, num_items, tl, vl, tsl, device, False, False, False),
        run_setting("DNMR-MemoryOnly", cfg, num_items, tl, vl, tsl, device, True, False, False),
        run_setting("DNMR-Full-v2", cfg, num_items, tl, vl, tsl, device, True, True, False),
    ]

def run_all_datasets(cfg, data_paths, device):
    all_results = {}
    for dataset_name in ["amazon_beauty", "amazon_sports", "ml1m"]:
        print("\n" + "=" * 80)
        print(f"RUNNING DATASET: {dataset_name}")
        print("=" * 80)
        all_results[dataset_name] = run_all_methods_on_dataset(dataset_name, cfg, data_paths, device)
    return all_results

## 6. Run all 3 datasets and summarize

In [ ]:
all_results = run_all_datasets(cfg=cfg, data_paths=DATA_PATHS, device=device)


RUNNING DATASET: amazon_beauty
Detected 2023 Amazon review format in Beauty_and_Personal_Care.jsonl.gz
  Loaded 16442065 positive interactions from Beauty_and_Personal_Care.jsonl.gz
[amazon_beauty] Train examples=1798947, val users=371647, test users=371647
Epoch 01 | loss=0.7627 | rel=1.0000 | amb=0.0000 | accept=1.0000 | queue=0 | Val NDCG@10=0.2832 | Val HR@10=0.4544
Epoch 02 | loss=0.4546 | rel=1.0000 | amb=0.0000 | accept=1.0000 | queue=0 | Val NDCG@10=0.2958 | Val HR@10=0.4785
Epoch 03 | loss=0.4426 | rel=1.0000 | amb=0.0000 | accept=1.0000 | queue=0 | Val NDCG@10=0.2964 | Val HR@10=0.4790
Epoch 04 | loss=0.4398 | rel=1.0000 | amb=0.0000 | accept=1.0000 | queue=0 | Val NDCG@10=0.2971 | Val HR@10=0.4804
Epoch 05 | loss=0.4376 | rel=1.0000 | amb=0.0000 | accept=1.0000 | queue=0 | Val NDCG@10=0.2977 | Val HR@10=0.4811
Epoch 06 | loss=0.4358 | rel=1.0000 | amb=0.0000 | accept=1.0000 | queue=0 | Val NDCG@10=0.2983 | Val HR@10=0.4819
Epoch 07 | loss=0.4350 | rel=1.0000 | amb=0.0000 | 

In [ ]:
rows = []
for dataset_name, dataset_results in all_results.items():
    for r in dataset_results:
        rows.append({
            "Dataset": dataset_name,
            "Model": r["name"],
            "HR@5": r["test_metrics"]["HR@5"],
            "HR@10": r["test_metrics"]["HR@10"],
            "HR@20": r["test_metrics"]["HR@20"],
            "NDCG@5": r["test_metrics"]["NDCG@5"],
            "NDCG@10": r["test_metrics"]["NDCG@10"],
            "NDCG@20": r["test_metrics"]["NDCG@20"],
        })

results_df = pd.DataFrame(rows).sort_values(["Dataset", "Model"]).reset_index(drop=True)
results_df.round(4)

,Dataset,Model,HR@5,HR@10,HR@20,NDCG@5,NDCG@10,NDCG@20
0,amazon_beauty,DNMR-Full-v2,0.3437,0.4598,0.5923,0.2470,0.2844,0.3179
1,amazon_beauty,DNMR-MemoryOnly,0.3423,0.4589,0.5924,0.2466,0.2842,0.3179
2,amazon_beauty,SASRec,0.3425,0.4600,0.5932,0.2462,0.2841,0.3177
3,amazon_sports,DNMR-Full-v2,0.2587,0.3635,0.4934,0.1816,0.2153,0.2481
4,amazon_sports,DNMR-MemoryOnly,0.2574,0.3624,0.4906,0.1809,0.2148,0.2471
5,amazon_sports,SASRec,0.2581,0.3640,0.4918,0.1802,0.2143,0.2465
6,ml1m,DNMR-Full-v2,0.4612,0.6238,0.7826,0.3175,0.3701,0.4103
7,ml1m,DNMR-MemoryOnly,0.4629,0.6226,0.7837,0.3217,0.3735,0.4143
8,ml1m,SASRec,0.4524,0.6117,0.7807,0.3098,0.3612,0.4040


## 7. Additional Baselines: BERT4Rec and GRU4Rec

- **GRU4Rec**: GRU-based sequential recommendation (Hidasi et al., 2016). Uses a single-layer GRU to encode the interaction sequence, then scores items via dot product.
- **BERT4Rec**: Bidirectional transformer for sequential recommendation (Sun et al., 2019). Uses masked item prediction (Cloze task) during training, causal scoring at inference.

In [3]:
class GRU4Rec(nn.Module):
    """GRU-based sequential recommendation (Hidasi et al., 2016)."""

    def __init__(self, num_items, cfg):
        super().__init__()
        self.cfg = cfg
        self.item_embedding = nn.Embedding(num_items + 1, cfg.embed_dim, padding_idx=cfg.padding_idx)
        self.gru = nn.GRU(cfg.embed_dim, cfg.embed_dim, num_layers=cfg.num_layers,
                          batch_first=True, dropout=cfg.dropout if cfg.num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(cfg.dropout)

    def encode_user(self, seq):
        """Encode user sequence with GRU, return last valid hidden state."""
        x = self.dropout(self.item_embedding(seq))  # (B, L, D)
        lengths = (seq != self.cfg.padding_idx).sum(dim=1).clamp(min=1)
        output, _ = self.gru(x)  # (B, L, D)
        # Gather last valid timestep
        idx = (lengths - 1).unsqueeze(1).unsqueeze(2).expand(-1, 1, output.size(2))
        return output.gather(1, idx).squeeze(1)  # (B, D)

    def score_items(self, user_repr, item_ids):
        item_emb = self.item_embedding(item_ids)
        if item_emb.dim() == 2:
            score = (user_repr * item_emb).sum(dim=-1)
        else:
            score = (user_repr.unsqueeze(1) * item_emb).sum(dim=-1)
        return score, item_emb

    def forward(self, seq, pos_items, neg_items):
        user_repr = self.encode_user(seq)
        pos_score, pos_emb = self.score_items(user_repr, pos_items)
        neg_score, neg_emb = self.score_items(user_repr, neg_items)
        return {"user_repr": user_repr, "pos_score": pos_score, "neg_score": neg_score,
                "pos_emb": pos_emb, "neg_emb": neg_emb}


print("GRU4Rec defined.")

GRU4Rec defined.


In [4]:
class BERT4Rec(nn.Module):
    """Bidirectional transformer for sequential recommendation (Sun et al., 2019).

    Training: randomly mask items in the sequence and predict them (Cloze task).
    Inference: append a [MASK] token at the end and predict the next item.
    """

    def __init__(self, num_items, cfg, mask_prob=0.2):
        super().__init__()
        self.cfg = cfg
        self.num_items = num_items
        self.mask_token = num_items + 1  # special MASK id
        self.mask_prob = mask_prob
        self.item_embedding = nn.Embedding(num_items + 2, cfg.embed_dim, padding_idx=cfg.padding_idx)  # +2 for mask
        self.pos_embedding = nn.Embedding(cfg.max_len, cfg.embed_dim)
        self.dropout = nn.Dropout(cfg.dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.embed_dim, nhead=cfg.num_heads, dim_feedforward=cfg.embed_dim * 4,
            dropout=cfg.dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=cfg.num_layers)
        self.output_layer = nn.Linear(cfg.embed_dim, num_items + 1)  # predict over real items (1..num_items)

    def _encode(self, seq):
        """Encode a (possibly masked) sequence."""
        bsz, seq_len = seq.shape
        pos_ids = torch.arange(seq_len, device=seq.device).unsqueeze(0).expand(bsz, -1)
        x = self.item_embedding(seq) + self.pos_embedding(pos_ids)
        x = self.dropout(x)
        # Padding mask: True where padded
        pad_mask = (seq == self.cfg.padding_idx)
        x = self.transformer(x, src_key_padding_mask=pad_mask)
        return x  # (B, L, D)

    def encode_user(self, seq):
        """For inference: append MASK token, encode, return MASK position output."""
        bsz = seq.size(0)
        # Trim last position and append mask token
        mask_col = torch.full((bsz, 1), self.mask_token, dtype=seq.dtype, device=seq.device)
        masked_seq = torch.cat([seq[:, 1:], mask_col], dim=1)  # keep same length
        encoded = self._encode(masked_seq)  # (B, L, D)
        return encoded[:, -1, :]  # last position = MASK

    def score_items(self, user_repr, item_ids):
        item_emb = self.item_embedding(item_ids)
        if item_emb.dim() == 2:
            score = (user_repr * item_emb).sum(dim=-1)
        else:
            score = (user_repr.unsqueeze(1) * item_emb).sum(dim=-1)
        return score, item_emb

    def forward(self, seq, pos_items, neg_items):
        """Training forward: mask random positions, compute CE loss on masked positions."""
        user_repr = self.encode_user(seq)
        pos_score, pos_emb = self.score_items(user_repr, pos_items)
        neg_score, neg_emb = self.score_items(user_repr, neg_items)
        return {"user_repr": user_repr, "pos_score": pos_score, "neg_score": neg_score,
                "pos_emb": pos_emb, "neg_emb": neg_emb}


print("BERT4Rec defined.")

BERT4Rec defined.


In [5]:
def train_baseline_one_epoch(model, loader, optimizer, device):
    """Standard BPR training loop for GRU4Rec / BERT4Rec (no memory)."""
    model.train()
    total_loss = 0.0
    for _, seq, pos, neg in loader:
        seq, pos, neg = seq.to(device), pos.to(device), neg.to(device)
        out = model(seq, pos, neg)
        loss = -F.logsigmoid(out["pos_score"] - out["neg_score"]).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(1, len(loader))


def fit_baseline(model, train_loader, val_loader, cfg, device):
    """Train a baseline model with early stopping on NDCG@10."""
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    best_val, best_state, patience_counter = -1.0, None, 0
    history = []
    for epoch in range(1, cfg.epochs + 1):
        loss = train_baseline_one_epoch(model, train_loader, optimizer, device)
        val_metrics = evaluate(model, val_loader, device)
        history.append({"epoch": epoch, "loss": loss, **val_metrics})
        print(f"Epoch {epoch:02d} | loss={loss:.4f} | Val NDCG@10={val_metrics['NDCG@10']:.4f} | Val HR@10={val_metrics['HR@10']:.4f}")
        if val_metrics["NDCG@10"] > best_val:
            best_val = val_metrics["NDCG@10"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= cfg.early_stopping_patience:
            print(f"Early stopping triggered at epoch {epoch}.")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return history


print("Baseline training functions defined.")

Baseline training functions defined.


In [ ]:
def run_additional_baselines(cfg, data_paths, device):
    """Run GRU4Rec and BERT4Rec on all three datasets."""
    baseline_results = {}
    for dataset_name in ["amazon_beauty", "amazon_sports", "ml1m"]:
        print("\n" + "=" * 80)
        print(f"BASELINES FOR: {dataset_name}")
        print("=" * 80)

        bundle = build_dataloaders_for_dataset(dataset_name, cfg, data_paths)
        num_items = bundle["num_items"]
        tl, vl, tsl = bundle["train_loader"], bundle["val_loader"], bundle["test_loader"]
        dataset_runs = []

        # --- GRU4Rec ---
        print("\n--- GRU4Rec ---")
        set_seed(42)
        gru_model = GRU4Rec(num_items=num_items, cfg=cfg).to(device)
        gru_history = fit_baseline(gru_model, tl, vl, cfg, device)
        gru_test = evaluate(gru_model, tsl, device)
        print(f"\nGRU4Rec TEST METRICS: {gru_test}")
        dataset_runs.append({"name": "GRU4Rec", "history": gru_history, "test_metrics": gru_test})

        # --- BERT4Rec ---
        print("\n--- BERT4Rec ---")
        set_seed(42)
        bert_model = BERT4Rec(num_items=num_items, cfg=cfg, mask_prob=0.2).to(device)
        bert_history = fit_baseline(bert_model, tl, vl, cfg, device)
        bert_test = evaluate(bert_model, tsl, device)
        print(f"\nBERT4Rec TEST METRICS: {bert_test}")
        dataset_runs.append({"name": "BERT4Rec", "history": bert_history, "test_metrics": bert_test})

        baseline_results[dataset_name] = dataset_runs

    return baseline_results


baseline_results = run_additional_baselines(cfg=cfg, data_paths=DATA_PATHS, device=device)


BASELINES FOR: amazon_beauty
Detected 2023 Amazon review format in Beauty_and_Personal_Care.jsonl.gz


KeyboardInterrupt: 

In [ ]:
# Combine original results with new baselines into one table
baseline_rows = []
for dataset_name, dataset_runs in baseline_results.items():
    for r in dataset_runs:
        baseline_rows.append({
            "Dataset": dataset_name,
            "Model": r["name"],
            "HR@5": r["test_metrics"]["HR@5"],
            "HR@10": r["test_metrics"]["HR@10"],
            "HR@20": r["test_metrics"]["HR@20"],
            "NDCG@5": r["test_metrics"]["NDCG@5"],
            "NDCG@10": r["test_metrics"]["NDCG@10"],
            "NDCG@20": r["test_metrics"]["NDCG@20"],
        })

baseline_df = pd.DataFrame(baseline_rows)

# Merge with original results_df
combined_df = pd.concat([results_df, baseline_df], ignore_index=True)
combined_df = combined_df.sort_values(["Dataset", "Model"]).reset_index(drop=True)
print("All models combined:")
combined_df.round(4)